### Load the Dataset (Keras Built-in)

In [1]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
VOCAB_SIZE = 10000
MAX_LEN = 100


In [3]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE
)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### What does the raw data look like?

In [4]:
print(x_train[0])
print(y_train[0])


[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
1


### Padding (VERY IMPORTANT)
Transformer needs fixed-length input.

In [5]:
x_train = pad_sequences(x_train, maxlen=MAX_LEN)
x_test = pad_sequences(x_test, maxlen=MAX_LEN)


### Full Transformer Model

In [6]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np


### Positional Encoding

In [7]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_encoding = self._positional_encoding(max_len, d_model)

    def _positional_encoding(self, position, d_model):
        angles = np.arange(position)[:, None] / np.power(
            10000, (2 * (np.arange(d_model)[None, :] // 2)) / d_model
        )
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        return tf.cast(angles[None, ...], tf.float32)

    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]


### Transformer Block

In [8]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model
        )
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model)
        ])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):
        attn = self.att(x, x)
        x = self.norm1(x + attn)
        ffn = self.ffn(x)
        return self.norm2(x + ffn)


### Build the Model

In [9]:
def build_model():
    inputs = layers.Input(shape=(MAX_LEN,))
    x = layers.Embedding(VOCAB_SIZE, 128)(inputs)
    x = PositionalEncoding(MAX_LEN, 128)(x)
    x = TransformerBlock(128, 4, 256)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs)


### Compile & Train

In [10]:
model = build_model()

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, 100, 128)       │             0 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 100, 128)       │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,618,561 (6.17 MB)

 Trainable params: 1,618,561 (6.17 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64
)


Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.5813 - loss: 0.6621 - val_accuracy: 0.8304 - val_loss: 0.3829
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8727 - loss: 0.3069 - val_accuracy: 0.8550 - val_loss: 0.3413
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9235 - loss: 0.2075 - val_accuracy: 0.8444 - val_loss: 0.3815
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9530 - loss: 0.1397 - val_accuracy: 0.8366 - val_loss: 0.4448
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9686 - loss: 0.0996 - val_accuracy: 0.8338 - val_loss: 0.5389
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9775 - loss: 0.0725 - val_accuracy: 0.8260 - val_loss: 0.5955
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9812 - loss: 0.0582 - val_accuracy: 0.8280 - val_loss: 0.6015
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9872 - loss: 0.0433 - val_acc

###How to Understand This as a Beginner
```Data flow:
Text review
 ↓
Token IDs
 ↓
Embedding
 ↓
Self-Attention (Transformer)
 ↓
Pooling
 ↓
Prediction (Positive / Negative)
```

In [12]:
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_accuracy)


782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7922 - loss: 1.1888
Test Accuracy: 0.7925199866294861


### Make Predictions

In [13]:
predictions = model.predict(x_test[:5])
print(predictions)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
[[0.95927745]
 [0.9998317 ]
 [0.9994098 ]
 [0.2350713 ]
 [0.999781  ]]


### Convert Probabilities to Labels

In [14]:
predicted_labels = (predictions > 0.5).astype(int)
print(predicted_labels)


[[1]
 [1]
 [1]
 [0]
 [1]]


### Compare Prediction vs Actual

In [15]:
for i in range(5):
    print(f"Predicted: {predicted_labels[i][0]}, Actual: {y_test[i]}")


Predicted: 1, Actual: 0
Predicted: 1, Actual: 1
Predicted: 1, Actual: 1
Predicted: 0, Actual: 0
Predicted: 1, Actual: 1


### Accuracy from Predictions

In [16]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test[:5], predicted_labels)
print("Manual Accuracy:", accuracy)


Manual Accuracy: 0.8


### Predict on NEW Text (Real Use Case)

In [17]:
from tensorflow.keras.datasets import imdb

word_index = imdb.get_word_index()

def encode_review(text):
    words = text.lower().split()
    encoded = [word_index.get(word, 2) for word in words]
    return pad_sequences([encoded], maxlen=MAX_LEN)


1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [26]:
review = "bad movie"
encoded_review = encode_review(review)

prediction = model.predict(encoded_review)
print("Positive probability:", prediction[0][0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Positive probability: 0.1152016


In [27]:
if prediction[0][0] > 0.5:
    print("Positive Review")
else:
    print("Negative Review")


Negative Review


In [29]:
tests = [
    "this movie is great",
    "this movie is bad",
    "this movie sucks",
    "this movie was terrible",
    "this movie was amazing"
]

for t in tests:
    p = model.predict(encode_review(t))[0][0]
    print(t, "→", p) # this predicts wrong as it transformer are data hungry and this model is simple


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
this movie is great → 0.90515774
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
this movie is bad → 0.00088567834
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
this movie sucks → 0.9995485
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
this movie was terrible → 0.98266745
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
this movie was amazing → 0.003968365


In [ ]:
t